# NC3 — GM & Lansing Findings Correlation
## Combined Institutional Footprint and Resident Outcomes Analysis
**Date:** November 04, 2026  
**Prepared for:** Lansing Growth Fund  
**Focus:** Integrating GM employment, tax base, and property data with tri-county resident and labor-market indicators

This notebook combines the GM-specific extracts (ACFR principal employers/taxpayers, parcel taxable values) with the broader Lansing findings (ACS resident outcomes, QCEW employment by ownership and industry) and examines descriptive correlations across the tri-county region.

Ultium Cells / LG Energy Solution is tracked separately and excluded from pure GM totals. All figures are drawn from the project's `Data_Files/` extracts.

In [ ]:
import pandas as pd
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
import warnings
warnings.filterwarnings('ignore')

pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 1000)
pd.set_option('display.float_format', '{:,.1f}'.format)

print('Environment ready — pandas + plotly')

## 1. GM Footprint Summary

Regional employment (ACFR Table 15) and City of Lansing taxable value (ACFR Table 7).

In [ ]:
# GM employment
gm_emp = pd.DataFrame({
    'Year': [2016, 2025],
    'Employees': [5222, 4549],
    'Pct of Metro Employment': [2.28, 2.15],
    'Metro Total': [229466, 211208],
    'Rank': [4, 4]
})
print('GM Regional Employment (Greater Lansing Region — ACFR Table 15)')
display(gm_emp)

# GM taxable value
gm_tax = pd.DataFrame({
    'Year': [2016, 2025],
    'Entity': ['General Motors LLC', 'GM LLC + GM Co. (combined)'],
    'Taxable Assessed Value': [59302202, 49069602 + 19921817],
    'Notes': ['Rank 1, 2.74% of city', 'Ranks 3 & 9; ~2.1% combined']
})
print('\nGM City of Lansing Taxable Value (ACFR Table 7)')
display(gm_tax)

print(f'\nEmployment change 2016→2025: {4549-5222:,} ({(4549/5222-1)*100:.1f}%)')
print(f'Taxable value change 2016→2025: ${(49069602+19921817)-59302202:,.0f} ({((49069602+19921817)/59302202-1)*100:.1f}%)')

## 2. Tri-County Resident & Employment Snapshot

Core ACS 2020–2024 and QCEW 2024 indicators for Clinton, Eaton, and Ingham.

In [ ]:
counties = pd.DataFrame({
    'Geography': ['Clinton County', 'Eaton County', 'Ingham County'],
    'Population': [79626, 109130, 283913],
    'Median HH Income': [88210, 79597, 65526],
    'Poverty Rate (%)': [9.7, 8.3, 16.7],
    'Owner-Occupied (%)': [82.2, 74.2, 55.0],
    'Mfg Employment (ACS)': [4506, 8573, 13549],
    'ACS Workers': [39229, 53315, 138634],
    'QCEW Total Employment': [18614, 44501, 155500],
    'QCEW State Gov': [15, 2351, 31363],
    'QCEW NAICS 336': [341, 4845, 3515]
})
counties['Mfg Share (%)'] = (counties['Mfg Employment (ACS)'] / counties['ACS Workers'] * 100).round(1)
counties['State Gov Share (%)'] = (counties['QCEW State Gov'] / counties['QCEW Total Employment'] * 100).round(1)
counties['NAICS 336 Share (%)'] = (counties['QCEW NAICS 336'] / counties['QCEW Total Employment'] * 100).round(1)

print('Tri-County Combined Snapshot')
display(counties[['Geography', 'Population', 'Median HH Income', 'Poverty Rate (%)',
                  'Mfg Share (%)', 'State Gov Share (%)', 'NAICS 336 Share (%)']])

## 3. City-Level Contrast

Lansing and East Lansing cities versus their counties and the MSA.

In [ ]:
cities = pd.DataFrame({
    'Geography': ['Lansing city', 'East Lansing city', 'Ingham County', 'MSA'],
    'Median HH Income': [54382, 45687, 65526, 72550],
    'Poverty Rate (%)': [20.1, 33.0, 16.7, 13.5],
    'Population': [113023, 45758, 283913, 472669]
})
print('City vs. Broader Geography')
display(cities)

fig = px.bar(
    cities, x='Geography', y='Median HH Income',
    title='Median Household Income — City vs. County vs. MSA',
    color='Geography', color_discrete_sequence=px.colors.qualitative.Set2
)
fig.update_layout(template='plotly_white', height=420, title_x=0.5, showlegend=False)
fig.show()

## 4. Cross-County Correlation Matrix

Descriptive correlations across the three counties. With n=3 these are directional only.

In [ ]:
corr_cols = ['Median HH Income', 'Poverty Rate (%)', 'Owner-Occupied (%)',
             'Mfg Share (%)', 'State Gov Share (%)', 'NAICS 336 Share (%)']
corr = counties[corr_cols].corr().round(2)
print('Cross-County Correlation Matrix (n=3, descriptive only)')
display(corr)

fig = px.imshow(
    corr, text_auto=True, color_continuous_scale='RdBu_r',
    zmin=-1, zmax=1, title='Correlation Heatmap — Tri-County Indicators'
)
fig.update_layout(height=500, title_x=0.5)
fig.show()

## 5. Institutional & Manufacturing Presence vs. Resident Outcomes

Scatter views linking employment composition to income and poverty. Eaton (strongest NAICS 336 / GM assembly presence) shows stronger county-level outcomes than Ingham (highest state-government share).

In [ ]:
fig = px.scatter(
    counties, x='State Gov Share (%)', y='Median HH Income',
    size='Population', color='Geography', text='Geography',
    title='State Government Employment Share vs. Median Household Income'
)
fig.update_traces(textposition='top center')
fig.update_layout(template='plotly_white', height=450, title_x=0.5)
fig.show()

fig2 = px.scatter(
    counties, x='NAICS 336 Share (%)', y='Poverty Rate (%)',
    size='Population', color='Geography', text='Geography',
    title='NAICS 336 (Transportation Equipment) Share vs. Poverty Rate'
)
fig2.update_traces(textposition='top center')
fig2.update_layout(template='plotly_white', height=450, title_x=0.5)
fig2.show()

## 6. GM Parcel Taxable Value Trend

Core real-property parcel for the Lansing Delta Township Assembly (8001 Davis Hwy, Eaton / PA 425).

In [ ]:
delta_tv = pd.DataFrame({
    'Tax Year': [2023, 2024, 2025, 2026],
    'Taxable Value': [31460656, 33033688, 34057732, 34977290]
})
print('Delta Township Assembly — Core Real Property Taxable Value')
display(delta_tv)

fig = px.line(
    delta_tv, x='Tax Year', y='Taxable Value', markers=True,
    title='Lansing Delta Township Assembly — Taxable Value Trend'
)
fig.update_layout(template='plotly_white', height=420, title_x=0.5)
fig.update_traces(line_color='#ff7f0e')
fig.show()

print(f'Change 2023→2026: +${34977290-31460656:,.0f} (+{(34977290/31460656-1)*100:.1f}%)')

## 7. Combined Correlation Takeaways

| Pattern | Observation | Implication for Partner Discussion |
|---------|-------------|------------------------------------|
| Institutional concentration | Ingham has the highest state-government employment share (MSU + state agencies) | Large public institutional footprint does not automatically produce higher resident median income |
| Manufacturing / GM presence | Eaton has the strongest NAICS 336 concentration (Delta Twp Assembly) and stronger county-level income/poverty outcomes | Manufacturing employment share aligns more closely with better county resident outcomes than state-government share |
| City–county gap | Lansing and East Lansing cities show substantially lower income and higher poverty than their counties | Mismatch is visible at city scale; tract-level analysis would refine neighborhood targeting |
| GM absolute vs. relative | Employment −13% since 2016; taxable value rank fallen from #1; core parcel TV still rising | Absolute footprint remains large, but relative contribution to metro employment and city tax base has declined |
| Ultium exclusion | Former GM JV now LG Energy Solution is a major taxpayer but excluded from GM totals | Partner should confirm whether the broader auto/EV cluster or pure GM footprint is the preferred frame |

**Caveats**
- Three-county correlations are directional only; do not treat coefficients as precise.
- QCEW single-employer cells are frequently suppressed; industry and ownership totals are the reliable public bounds.
- ACS (residence-based) and QCEW (workplace jobs) measure different concepts.
- City ACS figures include student populations that affect East Lansing statistics.
- Manufacturing personal property is largely exempt; real property and IFT parcels drive the taxable value series.

## 8. Suggested Next Steps

1. Move from three-county correlations to tract-level analysis using the ACS tract file already in `Data_Files/`.
2. Add QCEW wage distributions for NAICS 336 and state government as a wealth-retention proxy.
3. Overlay LODES origin-destination flows to test local capture of manufacturing and state-government jobs.
4. Confirm preferred geographic grain (city, county, or tract) and whether Ultium/LG should appear as a separate auto-cluster indicator.
5. Cross-check ACFR employment figures against any available Michigan Works or LEAP detailed employer files.

---
*Notebook prepared for the NC3 community partner meeting. Combines GM and Lansing findings; load the underlying CSVs for production refresh.*